# Income Classification with Logistic Regression – Solution

**Short name (GitHub):** `Income_LogReg`  
Use the skeleton to practice; this notebook is the worked key. Numbers below were produced with scikit-learn 1.x on `data/adult.data` (32,561 rows), `random_state=1`, `test_size=0.2`.

**Headline results (lesson spec, unscaled L1 C=0.05 liblinear)**

| Metric | Value |
|--------|-------|
| Class mix | 75.92% `<=50K` / 24.08% `>50K` |
| X shape after dummies | 32,561 × 24 |
| Intercept | ≈ −5.55 |
| Test accuracy | ≈ 0.827 |
| Test confusion | TN 4779, FP 247, FN 878, TP 609 |
| Precision / recall / F1 (`>50K`) | 0.71 / 0.41 / 0.52 |
| ROC AUC | ≈ 0.846 |
| L1 zeros | 6 of 24 coefficients |

## Inline cheat-sheet

Same table as the skeleton — see **`Income_LogReg_Cheatsheet.docx`**.

Lesson typo: `feature_cols` listed `hours-per-week` twice. Deduplicate. Cast dummies to `float` on sklearn ≥ 1.9.

## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load, name columns, strip whitespace

In [ ]:
col_names = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income",
]
df = pd.read_csv("data/adult.data", header=None, names=col_names)
for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].str.strip()
print(df.head())
print(df.shape)

## 2. EDA and assumptions

### Task 2.1 — class imbalance

≈ 76 / 24. A classifier that always predicts `<=50K` already scores 0.76 accuracy. Always report recall / AUC alongside accuracy.

In [ ]:
print(df.income.value_counts())
print(df.income.value_counts(normalize=True).round(4))

### Task 2.2 — dummy-encode (dedup first)

In [ ]:
feature_cols = [
    "age", "capital-gain", "capital-loss", "hours-per-week",
    "sex", "race", "hours-per-week", "education",
]
feature_cols_u = list(dict.fromkeys(feature_cols))  # keeps order, drops 2nd hours-per-week
X = pd.get_dummies(df[feature_cols_u], drop_first=True).astype(float)
print(feature_cols_u)
print(X.shape)
print(list(X.columns))

### Task 2.3 — heatmap

Education dummies are mutually exclusive so they are *negatively* correlated with each other, not 0.99 clones. Race dummies behave the same way. No pair requires an emergency drop the way `radius`/`perimeter`/`area` would.

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Feature correlation (dummy-encoded X)")
plt.tight_layout()
plt.show()

### Task 2.4 — scale diagnosis + y

`capital-gain` spans 0–99,999; `hours-per-week` spans 1–99; dummies are 0/1. Unscaled L1 still fits, but the penalty is *not* on a common scale, so continuous features look "small" in the coefficient bar chart even when they matter.

In [ ]:
for c in ["age", "capital-gain", "capital-loss", "hours-per-week"]:
    print(f"{c:16s} min={X[c].min():8.1f}  max={X[c].max():8.1f}  mean={X[c].mean():8.2f}")

y = np.where(df.income == "<=50K", 0, 1)
print("positivity rate", round(y.mean(), 4))

## 3. Fit the lesson model

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, random_state=1, test_size=0.2
)
log_reg = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
print(x_train.shape, x_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3))

### Task 3.2 — parameters

Intercept ≈ −5.55 (log-odds of >50K when every dummy is the dropped reference level and continuous features are 0 — not a realistic person, so do not over-read the intercept).

In [ ]:
print("Model Parameters, Intercept:")
print(log_reg.intercept_[0])
print("Model Parameters, Coeff:")
print(log_reg.coef_)

### Task 3.3 — confusion + accuracy

[[4779, 247], [878, 609]] on this split. Accuracy ≈ 0.827. The model is biased toward the majority class: 878 missed high earners vs 247 false alarms.

In [ ]:
print("Confusion Matrix on test set:")
print(confusion_matrix(y_test, y_pred))
print("Accuracy Score on test set:")
print(log_reg.score(x_test, y_test))
print(classification_report(y_test, y_pred, digits=3))

## 4. Coefficient table and bar plot

Largest **positive** log-odds shifts: `education_Prof-school`, `education_Doctorate`, `education_Masters`, `education_Bachelors`, `sex_Male`.

Largest **negative**: `education_7th-8th`, `education_11th`, `education_9th`, `race_Black`.

L1 zeros typically include several rare education/race levels (e.g. Preschool, 1st-4th, Other).

In [ ]:
coef_df = (
    pd.DataFrame({"var": x_train.columns, "coef": log_reg.coef_[0]})
    .query("coef.abs() > 0")
    .sort_values("coef")
)
print(coef_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(data=coef_df, x="var", y="coef", color="#2c7bb6")
plt.xticks(rotation=90)
plt.title("LR Coefficient Values")
plt.tight_layout()
plt.show()

## 5. ROC and AUC

AUC ≈ 0.85 — ranking quality is decent even though the default threshold under-detects the minority class.

In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print("ROC AUC score:", roc_auc)

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:, 1])
plt.figure()
plt.plot(fpr, tpr, color="darkorange", label="ROC curve (area = %0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], color="navy", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
plt.show()

## 6. Alternate code

### 6.1 Unique columns via Index

In [ ]:
X_alt = pd.get_dummies(
    df[pd.Index(feature_cols).unique()], drop_first=True
).astype(float)
print(X_alt.shape, list(X_alt.columns)[:8])
assert X_alt.shape[1] == X.shape[1]

### 6.2 Scaled L1 pipeline

Accuracy and AUC stay in the same neighbourhood. Coefficients are now per standard deviation, so `capital-gain` no longer looks "almost zero" just because its raw units are dollars.

In [ ]:
pipe = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(C=0.05, penalty="l1", solver="liblinear")),
])
pipe.fit(x_train, y_train)
p = pipe.predict_proba(x_test)[:, 1]
print("scaled acc", pipe.score(x_test, y_test))
print("scaled auc", roc_auc_score(y_test, p))
print(pd.DataFrame({
    "var": x_train.columns,
    "scaled_coef": pipe.named_steps["lr"].coef_[0],
}).sort_values("scaled_coef").to_string(index=False))

### 6.3 L2 default — no exact zeros

In [ ]:
log_l2 = LogisticRegression(max_iter=2000)
log_l2.fit(x_train, y_train)
print("L2 n_zero", int((np.abs(log_l2.coef_[0]) < 1e-12).sum()))
print("L2 acc", log_l2.score(x_test, y_test))
print("L2 auc", roc_auc_score(y_test, log_l2.predict_proba(x_test)[:, 1]))

### 6.4 education-num ordinal substitute

In [ ]:
feature_cols_ord = [
    "age", "capital-gain", "capital-loss", "hours-per-week",
    "sex", "race", "education-num",
]
X_ord = pd.get_dummies(df[feature_cols_ord], drop_first=True).astype(float)
xo_tr, xo_te, yo_tr, yo_te = train_test_split(X_ord, y, random_state=1, test_size=0.2)
lr_ord = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
lr_ord.fit(xo_tr, yo_tr)
print("ordinal acc", lr_ord.score(xo_te, yo_te))
print("ordinal AUC", roc_auc_score(yo_te, lr_ord.predict_proba(xo_te)[:, 1]))
print("education-num coef", lr_ord.coef_[0][list(xo_tr.columns).index("education-num")])

### 6.5 Threshold helper

In [ ]:
def predict_at(proba, t=0.5):
    return (proba >= t).astype(int)

p1 = y_pred_prob[:, 1]
print(f"{'t':>6} {'prec':>8} {'rec':>8} {'FP':>6} {'FN':>6} {'acc':>8}")
for t in (0.25, 0.35, 0.50, 0.65):
    pred = predict_at(p1, t)
    cm = confusion_matrix(y_test, pred)
    print(f"{t:6.2f} {precision_score(y_test, pred):8.3f} {recall_score(y_test, pred):8.3f} "
          f"{cm[0,1]:6d} {cm[1,0]:6d} {accuracy_score(y_test, pred):8.3f}")

## 7. More practice

### 7.1 Add marital-status

`Married-civ-spouse` is usually one of the strongest dummies in this extract — household structure proxies earnings better than race dummies.

In [ ]:
cols_m = feature_cols_u + ["marital-status"]
Xm = pd.get_dummies(df[cols_m], drop_first=True).astype(float)
xm_tr, xm_te, ym_tr, ym_te = train_test_split(Xm, y, random_state=1, test_size=0.2)
lr_m = LogisticRegression(C=0.05, penalty="l1", solver="liblinear")
lr_m.fit(xm_tr, ym_tr)
coef_m = pd.DataFrame({"var": xm_tr.columns, "coef": lr_m.coef_[0]}).sort_values("coef")
print(coef_m.reindex(coef_m.coef.abs().sort_values(ascending=False).index).head(8))
print("AUC with marital-status", roc_auc_score(ym_te, lr_m.predict_proba(xm_te)[:, 1]))

### 7.2 Balanced class weights

Recall of >50K rises; accuracy falls. Use this when missing a high earner is costlier than a false alarm.

In [ ]:
log_bal = LogisticRegression(
    C=0.05, penalty="l1", solver="liblinear", class_weight="balanced"
)
log_bal.fit(x_train, y_train)
pred_b = log_bal.predict(x_test)
print("balanced acc", accuracy_score(y_test, pred_b))
print("balanced recall", recall_score(y_test, pred_b))
print("balanced prec", precision_score(y_test, pred_b))
print("balanced auc", roc_auc_score(y_test, log_bal.predict_proba(x_test)[:, 1]))
print(confusion_matrix(y_test, pred_b))

### 7.3 Fairness slice on sex

In [ ]:
male = x_test["sex_Male"] == 1
p1 = y_pred_prob[:, 1]
pred = (p1 >= 0.5).astype(int)
print("n male / female in test", int(male.sum()), int((~male).sum()))
print("recall male  ", recall_score(y_test[male], pred[male]))
print("recall female", recall_score(y_test[~male], pred[~male]))
print("base rate male  ", y_test[male].mean())
print("base rate female", y_test[~male].mean())

### 7.4 Which metric when?

In [ ]:
premium_screen = (
    "Precision (and a higher threshold): a false >50K tag wastes underwriting / offer cost."
)
outreach = (
    "Recall (and a lower threshold): missing a true high earner means the program never reaches them."
)
exec_kpi = (
    "Accuracy is the familiar 'how often right' number but hide it behind the 76% majority baseline."
)
print(premium_screen)
print(outreach)
print(exec_kpi)

## 8. Simulation

In [ ]:
# --- editable parameters ---
C = 0.05
N = 8000
N_REPS = 12
NOISE = 0.00
T = 0.50
SEED = 1
# ---------------------------

rng = np.random.default_rng(SEED)
rows = []
for r in range(N_REPS):
    idx = rng.integers(0, len(X), size=N)
    Xs = X.iloc[idx].reset_index(drop=True)
    ys = y[idx].copy()
    xtr, xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=SEED + r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy()
        ytr[flip] = 1 - ytr[flip]
    m = LogisticRegression(C=C, penalty="l1", solver="liblinear", max_iter=2000)
    m.fit(xtr, ytr)
    p = m.predict_proba(xte)[:, 1]
    pred = (p >= T).astype(int)
    rows.append({
        "acc": accuracy_score(yte, pred),
        "recall": recall_score(yte, pred, zero_division=0),
        "prec": precision_score(yte, pred, zero_division=0),
        "auc": roc_auc_score(yte, p),
        "n_zero": int((np.abs(m.coef_[0]) < 1e-12).sum()),
    })
sim = pd.DataFrame(rows)
print(sim.describe().round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col in zip(axes, ["acc", "recall", "auc"]):
    ax.hist(sim[col], bins=8, color="#4c78a8", edgecolor="white")
    ax.set_title(col)
plt.suptitle(f"Income_LogReg simulation  C={C}  N={N}  noise={NOISE}  T={T}")
plt.tight_layout()
plt.show()

## 9. Audience rewrite

In [ ]:
expert = (
    "Unpenalized-scale L1-logistic (C=0.05, liblinear) on 24 dummy/continuous columns "
    "yields test AUC 0.846 and accuracy 0.827. At t=0.5 the positive-class recall is 0.41 "
    "(TP=609, FN=878). L1 zeros 6/24 coefficients; largest log-odds are professional-school "
    "and doctorate dummies. Coefficients on raw dollars are not comparable to 0/1 dummies — "
    "report scaled betas or odds ratios per meaningful unit. Sex and race are protected "
    "attributes; do not treat their coefficients as causal wage gaps."
)
technician = (
    "Strip strings, drop the duplicate hours-per-week, get_dummies(drop_first=True), "
    "cast to float. Fit LogisticRegression(C=0.05, penalty='l1', solver='liblinear') "
    "on an 80/20 seed=1 split. Save intercept, the non-zero coef table, confusion matrix, "
    "and ROC. If sklearn ≥1.8 warns that penalty is deprecated, the fit is still valid; "
    "the pipeline+StandardScaler cell is the production-safer variant."
)
executive = (
    "A simple 1994-census model sorts people by chance of earning above $50k reasonably well "
    "(AUC about 0.85) and is right on four out of five hold-out rows. It still misses about "
    "three in five of the actual high earners if we use the default 50% cutoff. Education and "
    "hours matter most. This is a screening score, not a current salary predictor and not a "
    "hiring rule."
)
nonspecialist = (
    "We asked a simple yes/no question: did this adult earn more than $50,000 in 1994? "
    "The model looks at age, extra investment income, hours worked, sex, race and school. "
    "It is better than a coin flip at ranking people, but if we force it to say yes or no "
    "it is cautious — it more often says 'probably not' because most people in the file "
    "earned $50k or less. Schooling past high school is the clearest plus."
)
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)

## 10. Good fit vs not a fit

In [ ]:
good_fit = [
    "1. Binary outcome with independent rows (one person, one census record).",
    "2. Need a sparse, readable scorecard (L1) rather than a black-box ranker.",
    "3. Teaching dummy encoding + class imbalance + ROC together.",
    "4. Baseline before trees / boosting on tabular socio-economic data.",
    "5. Underwriting / eligibility screens where odds ratios must be shown.",
    "6. Policy briefing that must contrast majority-class accuracy with minority recall.",
    "7. Feature-ablation studies (education dummies vs education-num).",
    "8. Threshold workshops with business owners (cost of FP vs FN).",
    "9. Fairness diagnostics when protected attributes are in the file.",
    "10. Monte-Carlo checks of C, n, and label noise before locking a spec.",
]
not_a_fit = [
    "1. Predicting *current* wages or 2020s income — the extract is 1994 USD and 1994 jobs.",
    "2. Causal claims ('being male raises income by β') — observational, collider-heavy.",
    "3. Multi-class or continuous income (use multinomial / tobit / quantile regression).",
    "4. Production scoring that must exclude protected attributes by regulation.",
    "5. Small-n problems that violate 10 events-per-variable after dummy explosion.",
]
for row in good_fit:
    print(row)
print("--- not a fit ---")
for row in not_a_fit:
    print(row)

## 11. Done checklist

- [x] 32,561 × 15 cleaned Adult table
- [x] 76/24 imbalance named
- [x] 24-column dummy matrix, heatmap, scale note
- [x] Lesson L1 model, CM [[4779,247],[878,609]], acc ≈ 0.827, AUC ≈ 0.846
- [x] Sparse coef bar + ROC
- [x] Alternates: unique-index, scaled pipeline, L2, education-num, threshold sweep
- [x] Practice: marital-status, class_weight, sex-slice, metric choice
- [x] Simulation knobs C / N / noise / T
- [x] Four-audience rewrite + good-fit list